# MaxEnt lifetime MEM: Python API example

This notebook shows how to call the MaxEnt lifetime solver directly from Python
using the convenience functions in `api.py`, without going through the GUI or CLI.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from chisurf.plugins.fluorescence_decay.maxent_decay.fmem.api import (
    run_lifetime_mem_from_arrays,
    build_tau_grid,
)

# --- 1. Build a simple synthetic decay + IRF ---
dt = 0.02  # ns per channel
n_channels = 512
t = np.arange(n_channels) * dt

# True lifetime and simple exponential decay (no background)
tau_true = 3.5
decay_ideal = np.exp(-t / tau_true)

# Simple Gaussian IRF around t = 1 ns
irf = np.exp(-0.5 * ((t - 1.0) / 0.1) ** 2)
irf /= irf.sum()

# Convolve by a simple numerical convolution to get a toy decay
decay_conv = np.convolve(decay_ideal, irf, mode="full")[:n_channels]

# Add Poisson noise
rng = np.random.default_rng(0)
scale = 2e4
decay_counts = rng.poisson(decay_conv * scale).astype(float)
irf_counts = irf * decay_counts.max() * 0.5

# --- 2. Run the MaxEnt lifetime solver via the API ---
tau_grid = build_tau_grid(tau_min=0.1, tau_max=8.0, tau_step=0.05)

result = run_lifetime_mem_from_arrays(
    decay=decay_counts,
    irf=irf_counts,
    dt=dt,
    tau=tau_grid,
    nu=1e-3,
    fit_start_fraction=0.9,
    optimize_nuisance=False,
)

# --- 3. Inspect the result ---
tau = np.asarray(result["tau"], dtype=float).ravel()
p = np.asarray(result["p"], dtype=float).ravel()
p /= p.sum() if p.sum() > 0 else 1.0

Fi = np.asarray(result["Fi"], dtype=float)
y_seg = np.asarray(result["y"], dtype=float).ravel()
sigma = np.asarray(result["sigma"], dtype=float).ravel()
fitstart, fitstop = result["fitrange"]

# Reconstruct fitted decay segment
fit_seg = (Fi @ p) * sigma
fit_full = np.zeros_like(decay_counts)
fit_full[fitstart : fitstop + 1] = fit_seg

# --- 4. Plot decay + fit and lifetime distribution ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].semilogy(t, np.maximum(decay_counts, 1.0), label="data")
axes[0].semilogy(t, np.maximum(fit_full, 1.0), label="MaxEnt fit")
axes[0].set_xlabel("time [ns]")
axes[0].set_ylabel("counts")
axes[0].legend()

axes[1].plot(tau, p, marker="o")
axes[1].set_xlabel("lifetime [ns]")
axes[1].set_ylabel("probability")
axes[1].set_title("Lifetime distribution p(τ)")

fig.tight_layout()
fig.show()
